In [2]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import pandas as pd

file_path = r"C:\VS Code\ProblemNo4_input.xlsx"
#----- Henter ut data -----#
df_raw = pd.read_excel(file_path, sheet_name=0, skiprows=4)
df_raw.columns = ['e_eng', 'S_eng']
df_raw = df_raw.dropna()
stress_eng = df_raw['S_eng'].values
elongation_eng = df_raw['e_eng'].values

#----- Gjør om til sann spenning og tøyning -----#
elongation_true = np.log(1 + elongation_eng)
stress_true = stress_eng * (1 + elongation_eng)

#-------------------------------------
# Plotting
#-------------------------------------

# ----- Plot for engineering spenning–tøyning ----- #
fig1, ax1 = plt.subplots(figsize=(7*1.6, 7))
ax1.plot(elongation_eng, stress_eng,color='red', label='Engineering spenning-tøyning')
ax1.set_xlabel('Engineering tøyning')
ax1.set_ylabel('Spenning (MPa)')
ax1.set_title('Al-1%Mg - Engineering spenning-tøyning')
ax1.legend()
ax1.grid(True)
plt.show()

# ----- Plot for sann spenning–tøyning ----- #
fig2, ax2 = plt.subplots(figsize=(7*1.6, 7))
ax2.plot(elongation_true, stress_true,color='blue', label='Sann spenning-tøyning')
ax2.set_xlabel('Sann tøyning')
ax2.set_ylabel('Spenning (MPa)')
ax2.set_title('Al-1%Mg - Sann spenning-tøyning')
ax2.legend()
ax2.grid(True)
plt.show()

# ----- Kombinert plot: engineering og sann spenning–tøyning ----- #
fig3, ax3 = plt.subplots(figsize=(7*1.6, 7))
ax3.plot(elongation_eng, stress_eng,color='red', label='Engineering spenning-tøyning')
ax3.plot(elongation_true, stress_true,color='blue', label='Sann spenning-tøyning')
ax3.set_xlabel('Tøyning')
ax3.set_ylabel('Spenning (MPa)')
ax3.set_title('Al-1%Mg - Engineering vs sann spenning-tøyning')
ax3.legend()
ax3.grid(True)
plt.show()


sample_elongation = elongation_true[::5]
sample_stress = stress_true[::5]

# ------ Filtrerer for å finne data før necking ----- #

index = np.argmax(sample_stress)

sample_elongation = sample_elongation[0:index]
sample_stress = sample_stress[0:index]

# ----- Finner parametere n og k ----- #

import numpy as np

def minste_kvadraters(listx, listy):
    x = np.asarray(listx)
    y = np.asarray(listy)
    A = np.column_stack((x, np.ones(len(x))))
    a, b = np.linalg.lstsq(A, y)[0]
    return a, b

a1,b1 =minste_kvadraters(np.log(sample_elongation),np.log(sample_stress))

# ------ Definerer funksjoner for hollomon og Voce ----- #

def hollomon(eps, K, n):
    return K * eps**n

def Voce(eps, sigma0, sigmas, elong_trans):
    return sigmas - (sigmas - sigma0) * np.exp(-eps / elong_trans)

# ------ Tilpasser kurver med curve_fit for å finne funksjonsvariabel ----- #

popt, idk, infodict, idk2, ier = curve_fit(hollomon,sample_elongation,sample_stress,p0=[200, 0.25],full_output=True)
K_fit, n_fit = popt

popt, pcov, infodict2, errmsg, ier = curve_fit(Voce,sample_elongation,sample_stress,p0=[20,200,0.01],full_output=True)
Sigma0, Sigmas, elong_trans = popt


# ------ Printer ut resultater ----- #
print("--------Log–log minste kvadrater------------")
print("n =", a1)
print("K =", np.exp(b1))
print("----------------Hollomon--------------------")
print("curve_fit: hollomon")
print("n =", n_fit)
print("K =", K_fit)
print("Antall funksjonsevalueringer:", infodict["nfev"])
print("----------------Voce------------------------")
print("curve_fit: Voce")
print("Sigma0 ", Sigma0)
print("Sigmas ", Sigmas)
print("Transiant elongation ", elong_trans)
print("Antall funksjonsevalueringer:", infodict2["nfev"])

#-------------------------------------
# Plotting
#-------------------------------------

fig, ax = plt.subplots(figsize=(7*1.6, 7))
ax.plot(elongation_true, stress_true,color='blue', label='Sann spenning')
ax.plot(elongation_true,hollomon(elongation_true, K_fit, n_fit),color='purple', label='Hollomon - curve_fit')
ax.plot(elongation_true,Voce(elongation_true, Sigma0, Sigmas, elong_trans),color='pink', label='Voce - curve_fit')
ax.plot(elongation_true,hollomon(elongation_true, np.exp(b1), a1),color='purple', linestyle='--',label='Hollomon - minste kvadraters metode (log-log)')
ax.set_xlabel('Sann tøyning [-]')
ax.set_ylabel('Spenning (MPa)')
ax.set_title('Al-1%Mg spenning–tøyningskurve med tilpassede modeller')
ax.legend()
ax.grid(True)
plt.show()

# ------ Sammenligning av Hollomon og Voce ved store tøyninger ----- #

# Lager tøyningsverdier fra 0 til 4
epsilon_extended = np.linspace(0, 4, 1000)
# Plotter begge modellene over større område
fig, ax = plt.subplots(figsize=(7*1.6, 7))
ax.plot(epsilon_extended, hollomon(epsilon_extended, K_fit, n_fit), color='purple', linewidth=2, label='Hollomon')
ax.plot(epsilon_extended, Voce(epsilon_extended, Sigma0, Sigmas, elong_trans), color='orange', linewidth=2, label='Voce')
ax.set_xlabel('Sann tøyning [-]')
ax.set_ylabel('Spenning (MPa)')
ax.set_title('Hollomon vs Voce ved store tøyninger (ε = 0-4)')
ax.legend()
ax.grid(False)
plt.show()
print("Done")




FileNotFoundError: [Errno 2] No such file or directory: 'C:\\VS Code\\ProblemNo4_input.xlsx'